# Losses Trial


In [1]:
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString, Polygon, box
import rasterio
import os
import matplotlib.pyplot as plt
import folium
from branca.colormap import LinearColormap
import os
import numpy as np
from shapely.ops import transform
import pyproj

In [2]:
root_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Analysis_Data")
traffic_data_path = root_dir.joinpath("traffic_werkdag_2024DEC.gpkg")
traffic_data = gpd.read_file(traffic_data_path, driver='GPKG')

netwerkschakel_data_path = Path(r"P:\bovenregionale-stresstest-hwn\Data\Shapes netwerkschakels\Shapes netwerkschakels")
netwerkschakels = netwerkschakel_data_path.joinpath("HWN_netwerkindeling.shp")

1. Find traffic on the on- and offramps (which are on- and which are off ramps, and find the traffic on these)
2. Find which NWBs belong to which netwerkschakels
3. Find where NWB traffic data should be added or substracted

In [3]:
joined_network

NameError: name 'joined_network' is not defined

In [5]:
import networkx as nx
def cluster_connected(gdf):
    """
    Spatially cluster AFR ramps that are connected (touching).
    Returns a copy of gdf with a 'cluster_id' column.
    """
    import networkx as nx
    G = nx.Graph()
    for idx, geom in gdf.geometry.items():
        G.add_node(idx)
    for idx1, geom1 in gdf.geometry.items():
        for idx2, geom2 in gdf.geometry.items():
            if idx1 < idx2 and geom1.touches(geom2):
                G.add_edge(idx1, idx2)
    clusters = list(nx.connected_components(G))
    gdf = gdf.copy()
    gdf["cluster_id"] = -1
    for cluster_idx, cluster_nodes in enumerate(clusters):
        gdf.loc[gdf.index.isin(cluster_nodes), "cluster_id"] = cluster_idx
    return gdf

In [6]:
# Assuming traffic_data is a GeoDataFrame
network_gdf = traffic_data

# Filter for rows where 'vbn_oms_tx' contains either "OPR" or "AFR"
ramps_gdf = network_gdf[network_gdf["vbn_oms_tx"].str.contains("OPR|AFR", na=False)]

# Separate AFR and OPR into their own GeoDataFrames
afr_gdf = ramps_gdf[ramps_gdf["vbn_oms_tx"].str.contains("AFR", na=False)].copy()
opr_gdf = ramps_gdf[ramps_gdf["vbn_oms_tx"].str.contains("OPR", na=False)].copy()


afr_gdf_clustered = cluster_connected(afr_gdf)
opr_gdf_clustered = cluster_connected(opr_gdf)


In [ ]:
afr_gdf_clustered

,vbn_id,vbn_oms_tx,vbn_oms_bp,vbn_lengte,regio_cntr,regio_alt,nwb_ids,wegnrhmp_b,wegnrhmp_e,bpszijde_b,...,km_b,km_e,hm_midden,vbn_id_tgn,vias_baan,traffic_vbn_id,AL_D_WR,VRPC_D_WR,geometry,cluster_id
210,10000000002052601770,A15 c (AFR) HENDRIK IDO AMBACHT 21,A15 c (AFR) 72.091 - 72.075,16.0,RWS WZ,None,205260027,A15,A15,Li,...,72.091,72.075,72.083,None,3584,10000000002052601770,2382.0,11.0,"LINESTRING (102848.000 430245.000, 102853.000 ...",0
1058,10000000004665080130,N33 a (AFR) ASSEN 32,N33 a (AFR) 5.800 - 6.557,755.0,RWS NN,None,"465509038, 465509039, 601269337",N33,N33,Re,...,5.800,6.557,6.178,None,14637,10000000004665080130,835.0,5.0,"LINESTRING (232580.960 554658.549, 232591.015 ...",1
1164,10000000006000126760,A16 c (AFR) RANDWEG DORDRECHT 20 (wegdeel B),A16 c (AFR) 38.497 - 38.402 (wegdeel B),95.0,RWS WZ,None,600173857,A16,A16,Li,...,38.497,38.402,38.449,None,88229,10000000006000126760,10299.0,17.0,"LINESTRING (104149.360 420621.683, 104154.150 ...",2
1197,10000000006001390230,A15 c (AFR) PAPENDRECHT 23,A15 c (AFR) 78.778 - 78.475,304.0,RWS WZ,None,"600478822, 600478826",A15,A15,Li,...,78.778,78.475,78.626,None,106348,10000000006001390230,2074.0,18.0,"LINESTRING (108792.530 427967.969, 108791.201 ...",3
1226,10621741180000000000,A58 c (AFR) RITTHEM 40,A58 c (AFR) 170.801 - 170.442,340.0,RWS ZD,None,"62174081, 600943942",A58,A58,Li,...,170.801,170.442,170.627,None,211,10621741180000000000,2204.0,8.0,"LINESTRING (31087.155 387062.070, 31131.426 38...",4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9043,16005020516005020530,A58 a (AFR) MIDDELBURG-CENTRUM 39 (wegdeel B),A58 a (AFR) 167.732 - 167.778 (wegdeel B),46.0,RWS ZD,None,"601258913, 601258921",A58,A58,Re,...,167.732,167.778,167.755,None,None,16005020516005020530,NaN,NaN,"LINESTRING (33290.571 389113.985, 33286.480 38...",9
9044,16005020550661771880,A58 c (AFR) MIDDELBURG-CENTRUM 39,A58 c (AFR) 167.663 - 167.624,48.0,RWS ZD,None,601258923,A58,A58,Li,...,167.663,167.624,167.644,None,None,16005020550661771880,NaN,NaN,"LINESTRING (33382.262 388837.963, 33386.332 38...",6
9045,16005020556005020570,A58 c (AFR) MIDDELBURG-CENTRUM 39 (wegdeel B),A58 c (AFR) 167.663 - 167.616 (wegdeel B),45.0,RWS ZD,None,601258927,A58,A58,Li,...,167.663,167.616,167.640,None,None,16005020556005020570,NaN,NaN,"LINESTRING (33382.262 388837.963, 33383.574 38...",6
9047,16005077151342790750,A15 c (AFR) HAVENS 6200-7000 9,A15 c (AFR) 27.445 - 27.409,36.0,RWS WZ,None,601269269,A15,A15,Li,...,27.445,27.409,27.427,None,None,16005077151342790750,NaN,NaN,"LINESTRING (67502.775 439894.043, 67500.047 43...",34


In [28]:
# === AFR_GDF_CLUSTERED ===

# Group by 'wegnrhmp_b' and sum 'AL_D_WR'
afr_gdf_uitgaand = afr_gdf_clustered.groupby("wegnrhmp_b")["AL_D_WR"].sum().reset_index()

# Count unique 'cluster_id' per 'wegnrhmp_b'
afr_cluster_counts = afr_gdf_clustered.groupby("wegnrhmp_b")["cluster_id"].nunique().reset_index()
afr_cluster_counts.rename(columns={"cluster_id": "unique_clusters"}, inplace=True)

# Merge and calculate average
afr_gdf_uitgaand = afr_gdf_uitgaand.merge(afr_cluster_counts, on="wegnrhmp_b")
afr_gdf_uitgaand["average_per_afr"] = afr_gdf_uitgaand["AL_D_WR"] / afr_gdf_uitgaand["unique_clusters"]

# === OPR_GDF_CLUSTERED ===

# Group by 'wegnrhmp_b' and sum 'AL_D_WR'
opr_gdf_ingaand = opr_gdf_clustered.groupby("wegnrhmp_b")["AL_D_WR"].sum().reset_index()

# Count unique 'cluster_id' per 'wegnrhmp_b'
opr_cluster_counts = opr_gdf_clustered.groupby("wegnrhmp_b")["cluster_id"].nunique().reset_index()
opr_cluster_counts.rename(columns={"cluster_id": "unique_clusters"}, inplace=True)

# Merge and calculate average
opr_gdf_ingaand = opr_gdf_ingaand.merge(opr_cluster_counts, on="wegnrhmp_b")
opr_gdf_ingaand["average_per_opr"] = opr_gdf_ingaand["AL_D_WR"] / opr_gdf_ingaand["unique_clusters"]

opr_gdf_ingaand



,wegnrhmp_b,AL_D_WR,unique_clusters,average_per_opr
0,A1,338928.0,62,5466.580645
1,A10,369494.0,34,10867.470588
2,A12,546875.0,58,9428.879310
3,A13,119821.0,12,9985.083333
4,A14,6620.0,1,6620.000000
5,A15,419075.0,65,6447.307692
6,A16,242071.0,23,10524.826087
7,A17,51545.0,18,2863.611111
8,A18,23974.0,8,2996.750000
9,A2,532301.0,90,5914.455556


In [30]:
# Merge afr and opr dataframes on 'wegnrhmp_b'
merged_df = opr_gdf_ingaand.merge(
    afr_gdf_uitgaand[["wegnrhmp_b", "average_per_afr"]],
    on="wegnrhmp_b",
    how="inner"  # or 'outer' if you want to keep all rows
)

# Calculate the difference between opr and afr averages
merged_df["average_difference"] = merged_df["average_per_opr"] - merged_df["average_per_afr"]

# Final result
merged_df


,wegnrhmp_b,AL_D_WR,unique_clusters,average_per_opr,average_per_afr,average_difference
0,A1,338928.0,62,5466.580645,5452.906250,13.674395
1,A10,369494.0,34,10867.470588,11484.454545,-616.983957
2,A12,546875.0,58,9428.879310,9208.237288,220.642022
3,A13,119821.0,12,9985.083333,11039.500000,-1054.416667
4,A15,419075.0,65,6447.307692,7491.531250,-1044.223558
5,A16,242071.0,23,10524.826087,10319.769231,205.056856
6,A17,51545.0,18,2863.611111,2893.722222,-30.111111
7,A18,23974.0,8,2996.750000,3995.500000,-998.750000
8,A2,532301.0,90,5914.455556,5516.923913,397.531643
9,A20,207730.0,23,9031.739130,7699.807692,1331.931438


In [ ]:
import geopandas as gpd

# Assuming you have these GeoDataFrames:
# - traffic_data: all segments including ramps and highways
# - afr_gdf: off-ramps
# - opr_gdf: on-ramps
# - hwy_gdf: highway segments

# Ensure all GeoDataFrames use the same CRS
afr_gdf = afr_gdf.to_crs(traffic_data.crs)
opr_gdf = opr_gdf.to_crs(traffic_data.crs)
hwy_gdf = traffic_data[traffic_data["vbn_oms_tx"].str.contains("HWY", na=False)].copy()

# Find nearest highway segment for each ramp
afr_gdf["nearest_hwy_id"] = afr_gdf.geometry.apply(
    lambda ramp: hwy_gdf.distance(ramp).idxmin()
)
opr_gdf["nearest_hwy_id"] = opr_gdf.geometry.apply(
    lambda ramp: hwy_gdf.distance(ramp).idxmin()
)

# Now you can group ramps by their nearest highway segment
# and sum traffic volumes to estimate entry/exit per segment
afr_grouped = afr_gdf.groupby("nearest_hwy_id")["traffic_volume"].sum()
opr_grouped = opr_gdf.groupby("nearest_hwy_id")["traffic_volume"].sum()

# Merge with highway segments
hwy_gdf["afr_traffic"] = hwy_gdf.index.map(afr_grouped).fillna(0)
hwy_gdf["opr_traffic"] = hwy_gdf.index.map(opr_grouped).fillna(0)

# Estimate continuity ratio per segment
hwy_gdf["continue_ratio"] = hwy_gdf["traffic_volume"] / (
    hwy_gdf["traffic_volume"] + hwy_gdf["afr_traffic"] + hwy_gdf["opr_traffic"]
)